In [1]:
# Source
datadownloadpath_Source = '/mnt/controller_data/disk1/zeju/fomo-task2'
datasavepath_Source = './Dataset_Meningioma/'

In [2]:
import glob
import tqdm
import SimpleITK as sitk
import numpy as np
import pandas as pd
import random
from collections import Counter
from scipy.ndimage import label, binary_fill_holes
import os
import shutil

# Source Date Preprocessing

In [3]:
# Image example
# /mnt/controller_data/disk1/zeju/fomo-task2/preprocessed/sub_1/ses_1/
# dwi_b1000.nii.gz
# flair.nii.gz
# swi.nii.gz
# Label example
# /mnt/controller_data/disk1/zeju/fomo-task2/labels/sub_1/ses_1/
# seg.nii.gz

imgs_flair = glob.glob(datadownloadpath_Source + "/preprocessed/*/*/flair.nii.gz")
imgs_flair = [ fid for fid in sorted(imgs_flair) ]
imgs_dwi = glob.glob(datadownloadpath_Source + "/preprocessed/*/*/dwi_b1000.nii.gz")
imgs_dwi = [ fid for fid in sorted(imgs_dwi) ]
imgs_swi = glob.glob(datadownloadpath_Source + "/preprocessed/*/*/swi.nii.gz")
imgs_swi = [ fid for fid in sorted(imgs_swi) ]
segs = glob.glob(datadownloadpath_Source + "/labels/*/*/seg.nii.gz")
segs = [ fid for fid in sorted(segs) ]

pids = [pid.split("/")[-3].split(".")[0].split("_")[-1] for pid in imgs_flair]

In [5]:
imgs_dwi

['/mnt/controller_data/disk1/zeju/fomo-task2/preprocessed/sub_1/ses_1/dwi_b1000.nii.gz',
 '/mnt/controller_data/disk1/zeju/fomo-task2/preprocessed/sub_10/ses_1/dwi_b1000.nii.gz',
 '/mnt/controller_data/disk1/zeju/fomo-task2/preprocessed/sub_11/ses_1/dwi_b1000.nii.gz',
 '/mnt/controller_data/disk1/zeju/fomo-task2/preprocessed/sub_12/ses_1/dwi_b1000.nii.gz',
 '/mnt/controller_data/disk1/zeju/fomo-task2/preprocessed/sub_13/ses_1/dwi_b1000.nii.gz',
 '/mnt/controller_data/disk1/zeju/fomo-task2/preprocessed/sub_14/ses_1/dwi_b1000.nii.gz',
 '/mnt/controller_data/disk1/zeju/fomo-task2/preprocessed/sub_15/ses_1/dwi_b1000.nii.gz',
 '/mnt/controller_data/disk1/zeju/fomo-task2/preprocessed/sub_16/ses_1/dwi_b1000.nii.gz',
 '/mnt/controller_data/disk1/zeju/fomo-task2/preprocessed/sub_17/ses_1/dwi_b1000.nii.gz',
 '/mnt/controller_data/disk1/zeju/fomo-task2/preprocessed/sub_18/ses_1/dwi_b1000.nii.gz',
 '/mnt/controller_data/disk1/zeju/fomo-task2/preprocessed/sub_19/ses_1/dwi_b1000.nii.gz',
 '/mnt/cont

In [6]:
segs

['/mnt/controller_data/disk1/zeju/fomo-task2/labels/sub_1/ses_1/seg.nii.gz',
 '/mnt/controller_data/disk1/zeju/fomo-task2/labels/sub_10/ses_1/seg.nii.gz',
 '/mnt/controller_data/disk1/zeju/fomo-task2/labels/sub_11/ses_1/seg.nii.gz',
 '/mnt/controller_data/disk1/zeju/fomo-task2/labels/sub_12/ses_1/seg.nii.gz',
 '/mnt/controller_data/disk1/zeju/fomo-task2/labels/sub_13/ses_1/seg.nii.gz',
 '/mnt/controller_data/disk1/zeju/fomo-task2/labels/sub_14/ses_1/seg.nii.gz',
 '/mnt/controller_data/disk1/zeju/fomo-task2/labels/sub_15/ses_1/seg.nii.gz',
 '/mnt/controller_data/disk1/zeju/fomo-task2/labels/sub_16/ses_1/seg.nii.gz',
 '/mnt/controller_data/disk1/zeju/fomo-task2/labels/sub_17/ses_1/seg.nii.gz',
 '/mnt/controller_data/disk1/zeju/fomo-task2/labels/sub_18/ses_1/seg.nii.gz',
 '/mnt/controller_data/disk1/zeju/fomo-task2/labels/sub_19/ses_1/seg.nii.gz',
 '/mnt/controller_data/disk1/zeju/fomo-task2/labels/sub_2/ses_1/seg.nii.gz',
 '/mnt/controller_data/disk1/zeju/fomo-task2/labels/sub_20/ses_1/s

In [7]:
pids

['1',
 '10',
 '11',
 '12',
 '13',
 '14',
 '15',
 '16',
 '17',
 '18',
 '19',
 '2',
 '20',
 '21',
 '22',
 '23',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9']

## Step 1, Data resample

In [8]:
# helper functions copy pasted
def resample_by_res(mov_img_obj, new_spacing, interpolator = sitk.sitkLinear, logging = True):
    resample = sitk.ResampleImageFilter()
    resample.SetInterpolator(interpolator)
    resample.SetOutputDirection(mov_img_obj.GetDirection())
    resample.SetOutputOrigin(mov_img_obj.GetOrigin())
    resample.SetUseNearestNeighborExtrapolator(True)
    mov_spacing = mov_img_obj.GetSpacing()

    resample.SetOutputSpacing(new_spacing)
    RES_COE = np.array(mov_spacing) * 1.0 / np.array(new_spacing)
    new_size = np.array(mov_img_obj.GetSize()) *  RES_COE 

    resample.SetSize( [int(sz+1) for sz in new_size] )
    if logging:
        print("Spacing: {} -> {}".format(mov_spacing, new_spacing))
        print("Size {} -> {}".format( mov_img_obj.GetSize(), new_size ))

    return resample.Execute(mov_img_obj)

def resample_lb_by_res(mov_lb_obj, new_spacing, interpolator = sitk.sitkLinear, ref_img = None, logging = True):
    src_mat = sitk.GetArrayFromImage(mov_lb_obj)
    lbvs = np.unique(src_mat)
    if logging:
        print("Label values: {}".format(lbvs))
    for idx, lbv in enumerate(lbvs):
        _src_curr_mat = np.float32(src_mat == lbv) 
        _src_curr_obj = sitk.GetImageFromArray(_src_curr_mat)
        _src_curr_obj.CopyInformation(mov_lb_obj)
        _tar_curr_obj = resample_by_res( _src_curr_obj, new_spacing, interpolator, logging )
        _tar_curr_mat = np.rint(sitk.GetArrayFromImage(_tar_curr_obj)) * lbv
        if idx == 0:
            out_vol = _tar_curr_mat
        else:
            out_vol[_tar_curr_mat == lbv] = lbv
    out_obj = sitk.GetImageFromArray(out_vol)
    out_obj.SetSpacing( _tar_curr_obj.GetSpacing() )
    if ref_img != None:
        out_obj.CopyInformation(ref_img)
    return out_obj

In [9]:
savefold = './Dataset_Meningioma/'
if os.path.exists(savefold) == False:
    os.mkdir(savefold)

targetspacing = [0.45, 0.45, 5.0]

In [34]:
for (imgfile, lblfile, pid) in zip(imgs_flair, segs, pids):
    
    saving_group = savefold + str(imgfile.split('/')[-2])
    if os.path.exists(saving_group) == False:
        os.mkdir(saving_group)
    
    saving_pid_group = saving_group + '/' + pid
    if os.path.exists(saving_pid_group) == False:
        os.mkdir(saving_pid_group)
    imgsavepath = saving_pid_group + '/image.nii.gz'
    lblsavepath = saving_pid_group + '/seg.nii.gz'
    
    img_obj = sitk.ReadImage( imgfile )
    seg_obj = sitk.ReadImage( lblfile )
    img_spa_ori = img_obj.GetSpacing()
    seg_obj.SetSpacing(img_spa_ori)

    array_lbl = sitk.GetArrayFromImage(seg_obj)
    print('Before resampling: label pixel number is:' + np.str_(np.sum(array_lbl)))

    res_img_o = resample_by_res(img_obj, targetspacing, interpolator = sitk.sitkLinear,
                                    logging = False)
    res_lb_o = resample_lb_by_res(seg_obj, targetspacing, interpolator = sitk.sitkLinear,
                                  ref_img = res_img_o, logging = False)
    print('After resampling: label pixel number is:' + np.str_(np.sum(sitk.GetArrayFromImage(res_lb_o))))
    sitk.WriteImage(res_img_o, imgsavepath, True) 
    sitk.WriteImage(res_lb_o, lblsavepath, True) 

Before resampling: label pixel number is:3695.0
After resampling: label pixel number is:4515.0
Before resampling: label pixel number is:562.0
After resampling: label pixel number is:2680.0
Before resampling: label pixel number is:1058.0
After resampling: label pixel number is:5585.0
Before resampling: label pixel number is:543.0
After resampling: label pixel number is:826.0
Before resampling: label pixel number is:229.0
After resampling: label pixel number is:216.0
Before resampling: label pixel number is:1689.0
After resampling: label pixel number is:2448.0
Before resampling: label pixel number is:85042.0
After resampling: label pixel number is:111143.0
Before resampling: label pixel number is:58.0
After resampling: label pixel number is:81.0
Before resampling: label pixel number is:283.0
After resampling: label pixel number is:269.0
Before resampling: label pixel number is:7908.0
After resampling: label pixel number is:38787.0
Before resampling: label pixel number is:1047.0
After res

## Step 2, Intensity Normalization

In [36]:
# get the volume list
imgs_volume = glob.glob(savefold + "*/*/image.nii.gz")
imgs_volume = [ fid for fid in sorted(imgs_volume) ]

segs_volume = glob.glob(savefold + "*/*/seg.nii.gz")
segs_volume = [ fid for fid in sorted(segs_volume) ]

pids_volume = [pids_volume.split("/")[-2] for pids_volume in imgs_volume]

In [37]:
for (imgfile, segfile, pid) in zip(imgs_volume, segs_volume, pids_volume):
    img_obj = sitk.ReadImage( imgfile )
    seg_obj = sitk.ReadImage( segfile )
    
    array = sitk.GetArrayFromImage(img_obj)

    pixel_mean = np.mean(array)
    pixel_std = np.std(array)
    array = (array - pixel_mean) / pixel_std

    normalized_img = sitk.GetImageFromArray(array)
    normalized_img.CopyInformation(img_obj)
    
    array = sitk.GetArrayFromImage(seg_obj)

    array[array > 0] = 1

    binary_seg = sitk.GetImageFromArray(array)
    binary_seg.CopyInformation(seg_obj)
    
    sitk.WriteImage(normalized_img, imgfile, True)
    sitk.WriteImage(binary_seg, segfile, True)
    print('Id ' + pid + ' normalized')

Id 1 normalized
Id 10 normalized
Id 11 normalized
Id 12 normalized
Id 13 normalized
Id 14 normalized
Id 15 normalized
Id 16 normalized
Id 17 normalized
Id 18 normalized
Id 19 normalized
Id 2 normalized
Id 20 normalized
Id 21 normalized
Id 22 normalized
Id 23 normalized
Id 3 normalized
Id 4 normalized
Id 5 normalized
Id 6 normalized
Id 7 normalized
Id 8 normalized
Id 9 normalized


## Step 2, Generate datafiles.

In [15]:
datafilepath = './datafile/Dataset_Meningioma/'
if os.path.exists(datafilepath) == False:
    os.mkdir(datafilepath)
if os.path.exists(datafilepath + '/train') == False:
    os.mkdir(datafilepath + '/train')
if os.path.exists(datafilepath + '/val') == False:
    os.mkdir(datafilepath + '/val')

In [16]:
domainlists = glob.glob(datasavepath_Source + 'ses_1/*/')

In [17]:
sorted(domainlists)

['./Dataset_Meningioma/ses_1/1/',
 './Dataset_Meningioma/ses_1/10/',
 './Dataset_Meningioma/ses_1/11/',
 './Dataset_Meningioma/ses_1/12/',
 './Dataset_Meningioma/ses_1/13/',
 './Dataset_Meningioma/ses_1/14/',
 './Dataset_Meningioma/ses_1/15/',
 './Dataset_Meningioma/ses_1/16/',
 './Dataset_Meningioma/ses_1/17/',
 './Dataset_Meningioma/ses_1/18/',
 './Dataset_Meningioma/ses_1/19/',
 './Dataset_Meningioma/ses_1/2/',
 './Dataset_Meningioma/ses_1/20/',
 './Dataset_Meningioma/ses_1/21/',
 './Dataset_Meningioma/ses_1/22/',
 './Dataset_Meningioma/ses_1/23/',
 './Dataset_Meningioma/ses_1/3/',
 './Dataset_Meningioma/ses_1/4/',
 './Dataset_Meningioma/ses_1/5/',
 './Dataset_Meningioma/ses_1/6/',
 './Dataset_Meningioma/ses_1/7/',
 './Dataset_Meningioma/ses_1/8/',
 './Dataset_Meningioma/ses_1/9/']

## Step 3, Process dwi images.

In [18]:
for (imgfile, pid) in zip(imgs_dwi, pids):
    
    saving_group = savefold + str(imgfile.split('/')[-2])
    if os.path.exists(saving_group) == False:
        os.mkdir(saving_group)
    
    saving_pid_group = saving_group + '/' + pid
    if os.path.exists(saving_pid_group) == False:
        os.mkdir(saving_pid_group)
    imgsavepath = saving_pid_group + '/image_c2.nii.gz'
    
    img_obj = sitk.ReadImage( imgfile )
    img_spa_ori = img_obj.GetSpacing()

    res_img_o = resample_by_res(img_obj, targetspacing, interpolator = sitk.sitkLinear,
                                    logging = False)
    sitk.WriteImage(res_img_o, imgsavepath, True)
    print('Id ' + pid + ' normalized')

Id 1 normalized
Id 10 normalized
Id 11 normalized
Id 12 normalized
Id 13 normalized
Id 14 normalized
Id 15 normalized
Id 16 normalized
Id 17 normalized
Id 18 normalized
Id 19 normalized
Id 2 normalized
Id 20 normalized
Id 21 normalized
Id 22 normalized
Id 23 normalized
Id 3 normalized
Id 4 normalized
Id 5 normalized
Id 6 normalized
Id 7 normalized
Id 8 normalized
Id 9 normalized


In [19]:
# get the volume list
imgs_volume = glob.glob(savefold + "*/*/image_c2.nii.gz")
imgs_volume = [ fid for fid in sorted(imgs_volume) ]

segs_volume = glob.glob(savefold + "*/*/seg.nii.gz")
segs_volume = [ fid for fid in sorted(segs_volume) ]

pids_volume = [pids_volume.split("/")[-2] for pids_volume in imgs_volume]

In [20]:
for (imgfile, segfile, pid) in zip(imgs_volume, segs_volume, pids_volume):
    img_obj = sitk.ReadImage( imgfile )
    seg_obj = sitk.ReadImage( segfile )
    
    array = sitk.GetArrayFromImage(img_obj)

    pixel_mean = np.mean(array)
    pixel_std = np.std(array)
    array = (array - pixel_mean) / pixel_std

    normalized_img = sitk.GetImageFromArray(array)
    normalized_img.CopyInformation(img_obj)
    
    sitk.WriteImage(normalized_img, imgfile, True)
    print('Id ' + pid + ' normalized')

Id 1 normalized
Id 10 normalized
Id 11 normalized
Id 12 normalized
Id 13 normalized
Id 14 normalized
Id 15 normalized
Id 16 normalized
Id 17 normalized
Id 18 normalized
Id 19 normalized
Id 2 normalized
Id 20 normalized
Id 21 normalized
Id 22 normalized
Id 23 normalized
Id 3 normalized
Id 4 normalized
Id 5 normalized
Id 6 normalized
Id 7 normalized
Id 8 normalized
Id 9 normalized
